# Mini-Project : MCP + Agents IA avec Gemini — "Dev Assistant"

**Objectif :** construire une application agentique de bout en bout qui orchestre **plusieurs serveurs
MCP** via un LLM (Gemini), avec une politique pilotée par outils (c'est le LLM qui décide de la
prochaine action, pas un enchaînement codé en dur).

**Thème choisi : "Dev assistant"** — un agent qui explore un dépôt de code via le **filesystem MCP
server**, consulte son historique via le **git MCP server**, puis utilise un **serveur MCP personnalisé**
(linter + générateur de changelog) pour analyser et résumer les changements.

**Environnement cible :** Google Colab (fonctionne aussi en local si Node/npm et Python sont installés).

**Serveurs MCP utilisés (≥ 2 tiers requis) :**
1. `@modelcontextprotocol/server-filesystem` (npx, tiers) — lecture/écriture de fichiers dans le repo.
2. `mcp-server-git` (Python, tiers) — statut, log, diff du dépôt git.
3. `custom_ops` (FastMCP, écrit par nous) — `lint_python`, `generate_changelog`, `summarize_lines`.

## 1. Installation des dépendances (Colab)

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "mcp-server-git" \
  "fastmcp>=2.0.0"

**Notes**
- `langchain-google-genai` : intégration LangChain pour Gemini.
- `langchain-mcp-adapters` : client MCP compatible avec les outils LangChain.
- `mcp-server-git` : serveur MCP tiers (Python) exposant les opérations git (status, log, diff, ...).
- `fastmcp` : pour écrire notre propre serveur MCP (`custom_ops`).

## 2. Configuration de la clé API Gemini

In [ ]:
import os
from getpass import getpass

if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass('Entrez votre GOOGLE_API_KEY : ')

print('GOOGLE_API_KEY configurée :', bool(os.environ.get('GOOGLE_API_KEY')))

## 3. Vérification de Node/NPM

Beaucoup de serveurs MCP (dont `server-filesystem`) sont distribués en packages Node, lancés via `npx`.
Sur Colab, si Node/npm ne sont pas déjà installés, on les installe avec `apt-get`.

In [ ]:
import subprocess

def _check(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except Exception as e:
        return None

node_v = _check(['node', '--version'])
npx_v = _check(['npx', '--version'])
print('node:', node_v, '| npx:', npx_v)

In [ ]:
# A executer seulement si node/npx manquent (Colab) :
# !apt-get -qq update
# !apt-get -qq install -y nodejs npm
# !node --version
# !npx --version

## 4. Préparation d'un mini dépôt de démonstration

On crée un petit dépôt git avec un fichier Python volontairement imparfait (lignes trop longues, `TODO`
non traité) afin que l'agent ait quelque chose à explorer, à diffuser (git) et à analyser (linter).

In [ ]:
import subprocess
from pathlib import Path

WORKDIR = Path('/content/dev_assistant_repo') if Path('/content').exists() else Path.cwd() / 'dev_assistant_repo'
WORKDIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=WORKDIR):
    return subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True)

run(['git', 'init'])
run(['git', 'config', 'user.email', 'agent@example.com'])
run(['git', 'config', 'user.name', 'Dev Assistant'])

app_py = WORKDIR / 'app.py'
app_py.write_text(
    "def add(a, b):\n"
    "    return a + b\n"
    "\n"
    "def very_long_function_name_that_definitely_goes_over_the_line_length_limit(x, y, z):\n"
    "    # TODO: refactor this, it is way too long and needs a cleanup pass\n"
    "    return x + y + z\n",
    encoding='utf-8',
)

run(['git', 'add', '.'])
run(['git', 'commit', '-m', 'Initial commit: add app.py'])

app_py.write_text(app_py.read_text(encoding='utf-8') + "\ndef subtract(a, b):\n    return a - b\n", encoding='utf-8')
run(['git', 'add', '.'])
run(['git', 'commit', '-m', 'Add subtract() function'])

print('Depot pret dans :', WORKDIR)
print(run(['git', 'log', '--oneline']).stdout)

## 5. Serveur MCP personnalisé (`custom_ops`)

Écrit avec **FastMCP**. Il expose des outils utiles à un assistant de développement :
- `ping` : vérification de santé.
- `summarize_lines` : statistiques simples sur une liste de lignes.
- `lint_python` : mini-linter (lignes trop longues, `TODO` restants, code qui ne compile pas).
- `generate_changelog` : transforme un diff git brut en changelog Markdown lisible.

In [ ]:
from pathlib import Path
import textwrap

server_path = Path(WORKDIR).parent / 'custom_mcp_server.py'
server_path.write_text(textwrap.dedent('''
    from fastmcp import FastMCP
    from typing import Dict, List
    import ast

    mcp = FastMCP(name="custom_ops")

    @mcp.tool
    def ping() -> str:
        """Health check tool."""
        return "pong"

    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        """Return counts about a list of lines."""
        total = len(lines)
        nonempty = sum(1 for l in lines if l.strip())
        return {"total_lines": total, "nonempty_lines": nonempty}

    @mcp.tool
    def lint_python(code: str, max_line_length: int = 79) -> Dict[str, object]:
        """Very small Python linter: flags long lines, leftover TODOs and syntax errors."""
        lines = code.splitlines()
        long_lines = [i + 1 for i, l in enumerate(lines) if len(l) > max_line_length]
        todo_lines = [i + 1 for i, l in enumerate(lines) if "TODO" in l]
        syntax_ok = True
        syntax_error = None
        try:
            ast.parse(code)
        except SyntaxError as e:
            syntax_ok = False
            syntax_error = str(e)
        return {
            "long_lines": long_lines,
            "todo_lines": todo_lines,
            "syntax_ok": syntax_ok,
            "syntax_error": syntax_error,
        }

    @mcp.tool
    def generate_changelog(diff: str) -> str:
        """Turn a raw git diff/log into a short Markdown changelog."""
        added = [l[1:].strip() for l in diff.splitlines() if l.startswith("+") and not l.startswith("+++")]
        removed = [l[1:].strip() for l in diff.splitlines() if l.startswith("-") and not l.startswith("---")]
        bullets = []
        if added:
            bullets.append(f"### Added ({len(added)} lines)")
            bullets.extend(f"- `{l}`" for l in added[:10] if l)
        if removed:
            bullets.append(f"### Removed ({len(removed)} lines)")
            bullets.extend(f"- `{l}`" for l in removed[:10] if l)
        return "\\n".join(bullets) if bullets else "No changes detected."

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''), encoding="utf-8")

print("Wrote:", server_path)

## 6. Connexion aux serveurs MCP (`MultiServerMCPClient`)

On enregistre les 3 serveurs : `filesystem` (npx, tiers), `git` (`mcp-server-git`, tiers) et `custom_ops`
(notre serveur FastMCP).

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(WORKDIR)],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", str(WORKDIR)],
    },
    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(mcp_connections)
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

print(f'{len(tools)} outils charges :')
for t in tools:
    print(' -', t.name)

## 7. Construction de l'agent Gemini (LangGraph)

L'agent est un **ReAct agent** LangGraph : à chaque étape, Gemini décide lui-même quel outil appeler
(`filesystem_*`, `git_*`, `custom_ops_*`) et avec quels arguments — aucun enchaînement n'est codé en dur.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0)

system_prompt = (
    "You are a dev assistant agent. You have tools to read files (filesystem_*), "
    "inspect git history/diffs (git_*), and analyse code quality / summarize changes "
    "(custom_ops_*: lint_python, generate_changelog, summarize_lines). "
    "Always use tools to gather real information before answering; never guess file "
    "contents or git history."
)

agent = create_react_agent(llm, tools, prompt=system_prompt)

## 8. Exécution de tâches multi-outils

On pose des requêtes qui nécessitent de combiner plusieurs serveurs MCP, pour vérifier que l'agent
choisit lui-même la bonne séquence d'outils.

In [ ]:
def run_agent(query: str):
    result = agent.invoke({'messages': [('user', query)]})
    print(result['messages'][-1].content)
    return result

_ = run_agent(
    f"List the files in the repository at {WORKDIR}, read app.py, and lint it. "
    "Report any long lines, leftover TODOs, and whether it is syntactically valid."
)

In [ ]:
_ = run_agent(
    "Look at the git commit history of this repository (git log), get the diff of the "
    "most recent commit, and generate a short Markdown changelog summarizing what changed."
)

In [ ]:
_ = run_agent(
    "Read app.py from the repository, count how many non-empty lines it has using the "
    "summarize_lines tool, then lint it, and finally give me a one-paragraph code review "
    "summarizing the findings."
)

## 9. Conclusion

- **Composition d'outils** : l'agent Gemini a orchestré 3 serveurs MCP distincts (filesystem, git,
  custom_ops) en décidant lui-même, à chaque étape, quel outil invoquer selon la requête — aucune
  logique de routage codée en dur.
- **Serveur tiers vs serveur maison** : `server-filesystem` et `mcp-server-git` sont des serveurs MCP
  officiels/tiers réutilisés tels quels ; `custom_ops` est un serveur FastMCP développé spécifiquement
  pour le thème "dev assistant" (lint, changelog, statistiques de lignes).
- **Limites / pistes d'amélioration** : le linter est volontairement simple (longueur de ligne, TODO,
  validité syntaxique) — on pourrait le brancher sur `flake8`/`ruff` pour un vrai linting ; le
  générateur de changelog pourrait être enrichi avec une vraie compréhension sémantique du diff via le
  LLM plutôt qu'un simple parsing textuel.